In [ ]:
import datetime
import pandas as pd
import logging

In [ ]:
raw_df = pd.read_excel("/content/Bio data.xlsx", header=None)

In [ ]:
raw_df.head(20)

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,Emp Code : 513,NaN,NaN,NaN,Emp Name : Ramesh,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
3,NaN,Mo,Tu,We,Th,Fr,Sa,Su,Mo,Tu,...,Mo,Tu,We,Th,Fr,Sa,Su,Mo,Tu,We
4,In Time\nOut Time\nLate Mins\nEarly Dep\nWork ...,09:59\n20:05\n0\n0\n10:06\nP,09:55\n19:56\n0\n4\n10:01\nP,09:51\n19:50\n0\n10\n9:59\nP,09:55\n19:55\n0\n5\n10:00\nP,09:57\n19:56\n0\n4\n9:59\nP,09:45\n19:46\n0\n14\n10:01\nP,\n\n0\n0\n0:00\nWO-I,09:52\n19:52\n0\n8\n10:00\nP,10:11\n20:10\n11\n0\n9:59\nP,...,09:52\n18:52\n0\n68\n9:00\nP,\n\n0\n0\n0:00\nA,09:48\n19:48\n0\n12\n10:00\nP,09:41\n19:42\n0\n18\n10:01\nP,09:53\n19:52\n0\n8\n9:59\nP,09:48\n19:52\n0\n8\n10:04\nP,\n\n0\n0\n0:00\nWO-I,09:54\n19:53\n0\n7\n9:59\nP,09:49\n19:49\n0\n11\n10:00\nP,09:57\n19:55\n0\n5\n9:58\nP
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Emp Code : 515,NaN,NaN,NaN,Emp Name :Ram,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
9,NaN,Mo,Tu,We,Th,Fr,Sa,Su,Mo,Tu,...,Mo,Tu,We,Th,Fr,Sa,Su,Mo,Tu,We


In [ ]:
def calc_work_hours(in_t, out_t):
    if not in_t or not out_t or ":" not in str(in_t) or ":" not in str(out_t):
        return "0:00"
    try:
        # Updated to handle HH:MM:SS format found in the Excel
        fmt = "%H:%M:%S" if str(in_t).count(":") == 2 else "%H:%M"
        t1 = datetime.datetime.strptime(str(in_t), fmt)
        t2 = datetime.datetime.strptime(str(out_t), fmt)
        diff = t2 - t1
        sec = diff.total_seconds()
        if sec < 0: sec += 86400  # Handle overnight shifts
        return f"{int(sec//3600)}:{int((sec%3600)//60):02d}"
    except Exception as e:
        return "0:00"

In [ ]:
# Helper function to validate reported vs calculated hours
def validate_hours(w_hrs, calc_hrs):
    return w_hrs == calc_hrs

In [ ]:
current_emp_id = None
current_emp_name = None
current_department = None

In [ ]:
# Reset records for processing
records = []
sno = 1
year=2026
month=1

In [ ]:
# Re-processing with corrected logic to capture separate rows and valid calculations
records = []
sno = 1

# State variables
current_emp_id = None
current_emp_name = None
current_department = None

# Temporary storage for the current employee's grid
emp_data = {}

for idx, row in raw_df.iterrows():
    row_str = " ".join(row.astype(str))

    # 1. Capture Employee Header
    if "Emp Code :" in row_str:
        for val in row:
            text = str(val)
            if "Emp Code :" in text: current_emp_id = text.replace("Emp Code :", "").strip()
            if "Emp Name :" in text: current_emp_name = text.replace("Emp Name :", "").strip()
            if "Department :" in text: current_department = text.replace("Department :", "").strip()
        emp_data = {day: {} for day in range(1, 32)}
        continue

    label = str(row.iloc[0]).strip()
    target_labels = {"In Time": "in_time", "Out Time": "out_time", "Work Hrs": "work_hours", "Status": "status", "Late Mins": "late_min", "Early Dep": "early_min"}

    if label in target_labels:
        attr = target_labels[label]
        for day_num in range(1, 32):
            if day_num < len(row):
                val = row.iloc[day_num]
                if pd.notna(val):
                    if day_num not in emp_data: emp_data[day_num] = {}
                    emp_data[day_num][attr] = str(val).strip()

                    if label == "Status":
                        d_info = emp_data[day_num]
                        try:
                            date_obj = datetime.date(year, month, day_num)
                            in_t = d_info.get("in_time", "")
                            out_t = d_info.get("out_time", "")
                            w_hrs = d_info.get("work_hours", "00:00:00")

                            calc_hrs = calc_work_hours(in_t, out_t)

                            records.append({
                                "sno": sno,
                                "emp_id": current_emp_id,
                                "emp_name": current_emp_name,
                                "department": current_department,
                                "date": date_obj.strftime("%Y-%m-%d"),
                                "day": day_num,
                                "days": date_obj.strftime("%A"),
                                "in_time": in_t,
                                "out_time": out_t,
                                "late_min": d_info.get("late_min", "0"),
                                "early_min": d_info.get("early_min", "0"),
                                "work_hours": w_hrs,
                                "calculated_work_hrs": calc_hrs,
                                "hours_valid": (w_hrs == calc_hrs),
                                "status": d_info.get("status", "")
                            })
                            sno += 1
                        except ValueError:
                            continue

In [ ]:
final_df = pd.DataFrame(records)

In [ ]:
# Re-create the DataFrame and show the calculated columns
final_df = pd.DataFrame(records)

In [ ]:
display(final_df.head(20))

,sno,emp_id,emp_name,department,date,day,days,in_time,out_time,late_min,early_min,work_hours,calculated_work_hrs,hours_valid,status
0,1,507,None,Back Office,2026-01-01,1,Thursday,,,0,0,00:00:00,0:00,False,A
1,2,507,None,Back Office,2026-01-02,2,Friday,09:49:00,18:55:00,0,5,09:06:00,9:06,False,P
2,3,507,None,Back Office,2026-01-03,3,Saturday,09:36:00,19:00:00,0,0,09:24:00,9:24,False,P
3,4,507,None,Back Office,2026-01-04,4,Sunday,,,0,0,00:00:00,0:00,False,WO-I
4,5,507,None,Back Office,2026-01-05,5,Monday,09:41:00,16:00:00,0,180,06:19:00,6:19,False,P
5,6,507,None,Back Office,2026-01-06,6,Tuesday,09:39:00,18:42:00,0,18,09:03:00,9:03,False,P
6,7,507,None,Back Office,2026-01-07,7,Wednesday,09:51:00,18:58:00,0,2,09:07:00,9:07,False,P
7,8,507,None,Back Office,2026-01-08,8,Thursday,09:49:00,19:05:00,0,0,09:16:00,9:16,False,P
8,9,507,None,Back Office,2026-01-09,9,Friday,09:55:00,19:04:00,0,0,09:09:00,9:09,False,P
9,10,507,None,Back Office,2026-01-10,10,Saturday,09:52:00,18:59:00,0,1,09:07:00,9:07,False,P


In [ ]:
final_df['date'] = pd.to_datetime(final_df['date'], format='%Y-%m-%d')
final_df.to_csv("attendance_cleaned.csv", index=False)

KeyError: 'date'

In [ ]:
jpd = pd.read_csv("/content/attendance_cleaned.csv", parse_dates=['date'])

In [ ]:
jpd.head(30)

/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")
/usr/local/lib/python3.12/dist-packages/google/colab/_dataframe_summarizer.py:88: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  cast_date_col = pd.to_datetime(column, errors="coerce")


,sno,emp_id,emp_name,department,date,day,days,in_time,out_time,late_min,early_min,work_hours,calculated_work_hrs,hours_valid,status
0,1,507,NaN,Back Office,2026-01-01,1,Thursday,NaN,NaN,0,0,00:00:00,0:00,False,A
1,2,507,NaN,Back Office,2026-01-02,2,Friday,09:49:00,18:55:00,0,5,09:06:00,0:00,False,P
2,3,507,NaN,Back Office,2026-01-03,3,Saturday,09:36:00,19:00:00,0,0,09:24:00,0:00,False,P
3,4,507,NaN,Back Office,2026-01-04,4,Sunday,NaN,NaN,0,0,00:00:00,0:00,False,WO-I
4,5,507,NaN,Back Office,2026-01-05,5,Monday,09:41:00,16:00:00,0,180,06:19:00,0:00,False,P
5,6,507,NaN,Back Office,2026-01-06,6,Tuesday,09:39:00,18:42:00,0,18,09:03:00,0:00,False,P
6,7,507,NaN,Back Office,2026-01-07,7,Wednesday,09:51:00,18:58:00,0,2,09:07:00,0:00,False,P
7,8,507,NaN,Back Office,2026-01-08,8,Thursday,09:49:00,19:05:00,0,0,09:16:00,0:00,False,P
8,9,507,NaN,Back Office,2026-01-09,9,Friday,09:55:00,19:04:00,0,0,09:09:00,0:00,False,P
9,10,507,NaN,Back Office,2026-01-10,10,Saturday,09:52:00,18:59:00,0,1,09:07:00,0:00,False,P


In [ ]:
jpd[['calculated_work_hrs']]

,calculated_work_hrs
0,10:06
1,10:01
2,9:59
3,10:00
4,9:59
...,...
305,0:00
306,0:00
307,0:00
308,0:00


In [ ]:
def hhmm_to_minutes(hhmm):
    if not isinstance(hhmm, str) or ":" not in hhmm:
        return 0
    h, m = hhmm.split(":")
    return int(h) * 60 + int(m)

def normalize_work_hours_real(actual_minutes,
                              unit=60,
                              grace=5,
                              min_hours=1,
                              max_hours=10):
    """
    Industry-style normalization using bucket + grace.
    """
    # Safety cap (no overtime)
    actual_minutes = min(actual_minutes, max_hours * unit)

    base_bucket = (actual_minutes // unit) * unit
    next_bucket = base_bucket + unit

    # Distance to next bucket
    if next_bucket - actual_minutes <= grace:
        normalized_minutes = next_bucket
    else:
        normalized_minutes = base_bucket

    normalized_hours = normalized_minutes // unit

    # Enforce minimum payable
    if normalized_hours < min_hours:
        return 0

    return min(normalized_hours, max_hours)

def get_normalized_hours(hhmm_str):
    mins = hhmm_to_minutes(hhmm_str)
    return normalize_work_hours_real(mins)

In [ ]:
final_df['normalized_work_hrs'] = final_df['calculated_work_hrs'].apply(get_normalized_hours)
display(final_df[['emp_id', 'date', 'in_time', 'out_time', 'calculated_work_hrs', 'normalized_work_hrs']].head(20))

,emp_id,date,in_time,out_time,calculated_work_hrs,normalized_work_hrs
0,507,2026-01-01,,,0:00,0
1,507,2026-01-02,09:49:00,18:55:00,9:06,9
2,507,2026-01-03,09:36:00,19:00:00,9:24,9
3,507,2026-01-04,,,0:00,0
4,507,2026-01-05,09:41:00,16:00:00,6:19,6
5,507,2026-01-06,09:39:00,18:42:00,9:03,9
6,507,2026-01-07,09:51:00,18:58:00,9:07,9
7,507,2026-01-08,09:49:00,19:05:00,9:16,9
8,507,2026-01-09,09:55:00,19:04:00,9:09,9
9,507,2026-01-10,09:52:00,18:59:00,9:07,9


In [ ]:
final_df.to_csv('salary.csv', index=False)
print('Updated salary.csv with corrected hours.')

Updated salary.csv with corrected hours.


In [ ]:
adf=pd.read_csv("/content/salary.csv")

In [ ]:
grouped_df = (
    adf.groupby(["emp_id", "department"], as_index=False)
       .agg(
           total_worked_hours=("normalized_work_hrs", "sum")
       )
)

In [ ]:
grouped_df.head(9)

,emp_id,department,total_worked_hours
0,513,Back Office,255
1,515,"HR, IT, Admin, Operations",79
2,516,Sales,274
3,527,Default Dept.,0
4,538,Default Dept.,152
5,549,"HR, IT, Admin, Operations",77
6,550,Default Dept.,263
7,567,Default Dept.,70
8,577,Default Dept.,146


In [ ]:
grouped_df.shape

(9, 3)

In [ ]:
salarypd=pd.read_csv("/content/salary_bio.csv")

In [ ]:
payroll_df = grouped_df.merge(
    salarypd,
    on="emp_id",
    how="left"
)

In [ ]:
HOURS_PER_DAY = 10
# Correct way to get input
attended_days_input = input("Enter the number of working days: ")
TOTAL_DAYS = int(attended_days_input)

leave = input("Enter the number of leave days: ")
WEEKOFF_AND_PAID_LEAVE = int(leave)
# Calculation logic
Total_hours = TOTAL_DAYS * HOURS_PER_DAY

Enter the number of working days: 31
Enter the number of working days: 5


In [ ]:
payroll_df["total_days"] = TOTAL_DAYS
payroll_df["weekoff_paid_leave"] = WEEKOFF_AND_PAID_LEAVE

In [ ]:
payroll_df["total_expected_hours"] = ((TOTAL_DAYS * HOURS_PER_DAY)- (WEEKOFF_AND_PAID_LEAVE * HOURS_PER_DAY))

In [ ]:
payroll_df.loc[payroll_df["total_expected_hours"] < 0, "total_expected_hours"] = 0

In [ ]:
payroll_df["salary_per_hour"] = payroll_df["Salary"] / Total_hours

In [ ]:
payroll_df.loc[payroll_df["total_expected_hours"] == 0, "salary_per_hour"] = 0

In [ ]:
base_salary = (payroll_df["total_worked_hours"]* payroll_df["salary_per_hour"])

In [ ]:
fixed_bonus = WEEKOFF_AND_PAID_LEAVE * (payroll_df["salary_per_hour"] * HOURS_PER_DAY)

In [ ]:
leverage_diff = (payroll_df["total_expected_hours"] - payroll_df["total_worked_hours"]).clip(lower=0, upper=3)
leverage_amount = leverage_diff * payroll_df["salary_per_hour"]

In [ ]:
payroll_df["net_salary"] = (base_salary + fixed_bonus + leverage_amount)

In [ ]:
import numpy as np

payroll_df["deduction"] = np.select(
    [
        payroll_df["Salary"] <= 21000,
        (payroll_df["Salary"] >= 21001) & (payroll_df["Salary"] <= 24999),
        payroll_df["Salary"] >= 25000
    ],
    [
        payroll_df["Salary"] * 0.0075,  # ESI 0.75%
        0,                                  # No deduction
        200                                # Fixed PD
    ],
    default=0
)

In [ ]:
payroll_df["leverage_diff"]=leverage_diff
payroll_df["leverage_amount"]=leverage_amount
payroll_df["base_salary"]=base_salary
payroll_df["leave_amt"]=fixed_bonus
payroll_df["final_payable_salary"] = payroll_df["net_salary"] - payroll_df["deduction"]


In [ ]:
# Define the desired column order
column_order = [
    'emp_id',
    'Salary',
    'total_worked_hours',
    'total_expected_hours',
    'leverage_diff',
    'weekoff_paid_leave',
    'salary_per_hour',
        'base_salary',
    'leverage_amount',
    'net_salary',
    'deduction',
    'final_payable_salary'
]

# Filter the dataframe to only include these columns in this specific order
# We check if columns exist to avoid errors
existing_cols = [col for col in column_order if col in payroll_df.columns]
payroll_report = payroll_df[existing_cols].copy()

# Display the rearranged dataframe
display(payroll_report.head())

,emp_id,Salary,total_worked_hours,total_expected_hours,leverage_diff,weekoff_paid_leave,salary_per_hour,base_salary,leverage_amount,net_salary,deduction,final_payable_salary
0,513,20000,255,260,3,5,64.516129,16451.612903,193.548387,19870.967742,150.0,19720.967742
1,515,21000,79,260,3,5,67.741935,5351.612903,203.225806,8941.935484,157.5,8784.435484
2,516,20000,274,260,0,5,64.516129,17677.419355,0.000000,20903.225806,150.0,20753.225806
3,527,21000,0,260,3,5,67.741935,0.000000,203.225806,3590.322581,157.5,3432.822581
4,538,20000,152,260,3,5,64.516129,9806.451613,193.548387,13225.806452,150.0,13075.806452


In [ ]:
Salary calculation formulas

Common (Attendanace_Cleaning.ipynb, TPay.ipynb)
- total_expected_hours = (TOTAL_DAYS * HOURS_PER_DAY) - (WEEKOFF_AND_PAID_LEAVE * HOURS_PER_DAY)
- salary_per_hour = Salary / (TOTAL_DAYS * HOURS_PER_DAY)
- base_salary = total_worked_hours * salary_per_hour
- fixed_bonus = WEEKOFF_AND_PAID_LEAVE * (salary_per_hour * HOURS_PER_DAY)
- leverage_diff = clamp(total_expected_hours - total_worked_hours, 0, 3)
- leverage_amount = leverage_diff * salary_per_hour
- net_salary = base_salary + fixed_bonus + leverage_amount
- deduction =
  - if Salary <= 21000: Salary * 0.75
  - if 21001 <= Salary <= 24999: 0
  - if Salary >= 25000: 200
- final_payable_salary = net_salary - deduction

Payroll.ipynb (fixed salary)
- monthly_salary = FIXED_SALARY
- salary_per_hour = monthly_salary / (TOTAL_DAYS * HOURS_PER_DAY)
- base_salary = total_worked_hours * salary_per_hour
- fixed_bonus = WEEKOFF_AND_PAID_LEAVE * (salary_per_hour * HOURS_PER_DAY)
- leverage_diff = clamp(total_expected_hours - total_worked_hours, 0, 3)
- leverage_amount = leverage_diff * salary_per_hour
- net_salary = base_salary + fixed_bonus + leverage_amount

# Task
Process the attendance data from "/content/Bio data.xlsx" and "/content/salary_bio.csv" to generate a final payroll report. This includes:
1. Re-initializing necessary libraries and helper functions (`calc_work_hours`, `get_normalized_hours`).
2. Extracting 'In Time', 'Out Time', and 'Work Hrs' from the Excel file using the state-machine logic.
3. Calculating payable hours by applying industry-style normalization (bucket + grace) to the attendance records.
4. Performing payroll calculations: determining the salary per hour, base salary, fixed bonus (for week-offs/paid leaves), and leverage adjustments.
5. Calculating deductions based on salary brackets (ESI or PD).
6. Generating and displaying the final payroll report with the calculated net payable salary and saving it to 'salary.csv'.

## Re-initialize Dependencies

### Subtask:
Import required libraries and define helper functions for time calculation and normalization.


**Reasoning**:
Importing essential libraries and defining the core logic for time calculation and normalization as requested in the subtask instructions.



In [ ]:
import pandas as pd
import datetime
import logging
import numpy as np

def calc_work_hours(in_t, out_t):
    if not in_t or not out_t or ":" not in str(in_t) or ":" not in str(out_t):
        return "0:00"
    try:
        # Handle both HH:MM and HH:MM:SS formats
        fmt = "%H:%M:%S" if str(in_t).count(":") == 2 else "%H:%M"
        t1 = datetime.datetime.strptime(str(in_t).strip(), fmt)
        t2 = datetime.datetime.strptime(str(out_t).strip(), fmt)
        diff = t2 - t1
        sec = diff.total_seconds()
        if sec < 0: sec += 86400  # Handle overnight shifts
        return f"{int(sec//3600)}:{int((sec%3600)//60):02d}"
    except Exception as e:
        return "0:00"

def hhmm_to_minutes(hhmm):
    if not isinstance(hhmm, str) or ":" not in hhmm:
        return 0
    try:
        h, m = map(int, hhmm.split(":")[:2])
        return h * 60 + m
    except:
        return 0

def normalize_work_hours_real(actual_minutes, unit=60, grace=5, min_hours=1, max_hours=10):
    # Safety cap
    actual_minutes = min(actual_minutes, max_hours * unit)
    base_bucket = (actual_minutes // unit) * unit
    next_bucket = base_bucket + unit

    # Industry-style normalization using bucket + grace
    if next_bucket - actual_minutes <= grace:
        normalized_minutes = next_bucket
    else:
        normalized_minutes = base_bucket

    normalized_hours = normalized_minutes // unit
    return max(0, min(normalized_hours, max_hours)) if normalized_hours >= min_hours else 0

def get_normalized_hours(hhmm_str):
    mins = hhmm_to_minutes(hhmm_str)
    return normalize_work_hours_real(mins)

# Initialize state variables for data extraction
records = []
sno = 1
year = 2026
month = 1
print('Dependencies re-initialized and helper functions defined.')

Dependencies re-initialized and helper functions defined.


## Load and Process Raw Data

### Subtask:
Reload the attendance Excel file and execute the state-machine logic to extract clock-in and clock-out data into a list of records.


**Reasoning**:
I will implement the state-machine logic to extract employee-wise attendance data from the raw Excel file, processing headers and day-wise details into a structured list of records as specified.



In [ ]:
import pandas as pd
import datetime

# 1. Reload the file
raw_df = pd.read_excel("/content/Bio data.xlsx", header=None)

# 2. Re-initialize records and variables
records = []
sno = 1
year = 2026
month = 1

current_emp_id = None
current_emp_name = None
current_department = None
emp_data = {}

# 3. State-machine iteration
target_labels = {
    "In Time": "in_time",
    "Out Time": "out_time",
    "Work Hrs": "work_hours",
    "Status": "status",
    "Late Mins": "late_min",
    "Early Dep": "early_min"
}

for idx, row in raw_df.iterrows():
    row_str = " ".join(row.astype(str))

    # Capture Employee Header
    if "Emp Code :" in row_str:
        for val in row:
            text = str(val)
            if "Emp Code :" in text: current_emp_id = text.replace("Emp Code :", "").strip()
            if "Emp Name :" in text: current_emp_name = text.replace("Emp Name :", "").strip()
            if "Department :" in text: current_department = text.replace("Department :", "").strip()
        # Reset temporary grid for new employee
        emp_data = {day: {} for day in range(1, 32)}
        continue

    label = str(row.iloc[0]).strip()

    if label in target_labels:
        attr = target_labels[label]
        for day_num in range(1, 32):
            if day_num < len(row):
                val = row.iloc[day_num]
                if pd.notna(val):
                    if day_num not in emp_data: emp_data[day_num] = {}
                    emp_data[day_num][attr] = str(val).strip()

                    # When 'Status' (the last label in a block) is reached, commit the record
                    if label == "Status":
                        d_info = emp_data[day_num]
                        try:
                            date_obj = datetime.date(year, month, day_num)
                            in_t = d_info.get("in_time", "")
                            out_t = d_info.get("out_time", "")
                            w_hrs = d_info.get("work_hours", "00:00:00")

                            # 5. Calculate work hours
                            calc_hrs = calc_work_hours(in_t, out_t)

                            # 6. Append dictionary
                            records.append({
                                "sno": sno,
                                "emp_id": current_emp_id,
                                "emp_name": current_emp_name,
                                "department": current_department,
                                "date": date_obj.strftime("%Y-%m-%d"),
                                "day": day_num,
                                "days": date_obj.strftime("%A"),
                                "in_time": in_t,
                                "out_time": out_t,
                                "late_min": d_info.get("late_min", "0"),
                                "early_min": d_info.get("early_min", "0"),
                                "work_hours": w_hrs,
                                "calculated_work_hrs": calc_hrs,
                                "status": d_info.get("status", "")
                            })
                            sno += 1
                        except ValueError:
                            continue

print(f'Processed {len(records)} attendance records.')

Processed 0 attendance records.


**Reasoning**:
The previous attempt to extract data failed (0 records) because the 'label' variable was being matched against single labels like 'In Time', whereas the raw Excel data combines these labels into a single multi-line string (e.g., 'In Time\nOut Time...'). I will adjust the logic to handle these combined labels and split the multi-line cell values into individual components to correctly populate the 'records' list.



In [ ]:
import pandas as pd
import datetime

raw_df = pd.read_excel("/content/Bio data.xlsx", header=None)

records = []
sno = 1
year, month = 2026, 1
current_emp_id = None
current_emp_name = None
current_department = "Default Dept."

for idx, row in raw_df.iterrows():
    row_str = " ".join(row.astype(str))

    # 1. Identity Employee Header
    if "Emp Code :" in row_str:
        for val in row:
            text = str(val)
            if "Emp Code :" in text: current_emp_id = text.split(":")[-1].strip()
            if "Emp Name :" in text: current_emp_name = text.split(":")[-1].strip()
            if "Department :" in text: current_department = text.split(":")[-1].strip()
        continue

    # 2. Identify the Data Block (Labels are stacked in the first column)
    first_col = str(row.iloc[0])
    if "In Time" in first_col and "Status" in first_col:
        # Iterate through days (Columns 1 to 31)
        for day_num in range(1, 32):
            if day_num < len(row):
                val = str(row.iloc[day_num])
                if "\n" in val:
                    # Values are stacked: In\nOut\nLate\nEarly\nWork\nStatus
                    parts = [p.strip() for p in val.split("\n")]
                    if len(parts) >= 6:
                        in_t, out_t, late, early, w_hrs, status = parts[:6]

                        try:
                            date_obj = datetime.date(year, month, day_num)
                            calc_hrs = calc_work_hours(in_t, out_t)

                            records.append({
                                "sno": sno,
                                "emp_id": current_emp_id,
                                "emp_name": current_emp_name,
                                "department": current_department,
                                "date": date_obj.strftime("%Y-%m-%d"),
                                "day": day_num,
                                "days": date_obj.strftime("%A"),
                                "in_time": in_t,
                                "out_time": out_t,
                                "late_min": late,
                                "early_min": early,
                                "work_hours": w_hrs,
                                "calculated_work_hrs": calc_hrs,
                                "status": status
                            })
                            sno += 1
                        except ValueError:
                            continue

print(f'Successfully processed {len(records)} attendance records.')
if len(records) > 0: print(f'Sample: {records[0]["emp_id"]} on {records[0]["date"]}')

Successfully processed 310 attendance records.
Sample: 513 on 2026-01-01


## Generate Cleaned DataFrame

### Subtask:
Create a structured DataFrame from the extracted records, normalize the data types, and compute the industry-standard payable hours.


**Reasoning**:
I will convert the extracted records into a structured DataFrame, ensure correct date formatting, and apply the normalization logic to calculate payable hours as instructed.



In [ ]:
import pandas as pd

# 1. Convert records list to DataFrame
final_df = pd.DataFrame(records)

# 2. Convert date column to datetime objects
final_df['date'] = pd.to_datetime(final_df['date'], format='%Y-%m-%d')

# 3. Apply normalization to calculate payable hours
final_df['normalized_work_hrs'] = final_df['calculated_work_hrs'].apply(get_normalized_hours)

# 4. Verify the structure
print(f'DataFrame created with {len(final_df)} rows.')
display(final_df.head(20))

DataFrame created with 310 rows.


,sno,emp_id,emp_name,department,date,day,days,in_time,out_time,late_min,early_min,work_hours,calculated_work_hrs,status,normalized_work_hrs
0,1,513,Ramesh,Back Office,2026-01-01,1,Thursday,09:59,20:05,0,0,10:06,10:06,P,10
1,2,513,Ramesh,Back Office,2026-01-02,2,Friday,09:55,19:56,0,4,10:01,10:01,P,10
2,3,513,Ramesh,Back Office,2026-01-03,3,Saturday,09:51,19:50,0,10,9:59,9:59,P,10
3,4,513,Ramesh,Back Office,2026-01-04,4,Sunday,09:55,19:55,0,5,10:00,10:00,P,10
4,5,513,Ramesh,Back Office,2026-01-05,5,Monday,09:57,19:56,0,4,9:59,9:59,P,10
5,6,513,Ramesh,Back Office,2026-01-06,6,Tuesday,09:45,19:46,0,14,10:01,10:01,P,10
6,7,513,Ramesh,Back Office,2026-01-07,7,Wednesday,,,0,0,0:00,0:00,WO-I,0
7,8,513,Ramesh,Back Office,2026-01-08,8,Thursday,09:52,19:52,0,8,10:00,10:00,P,10
8,9,513,Ramesh,Back Office,2026-01-09,9,Friday,10:11,20:10,11,0,9:59,9:59,P,10
9,10,513,Ramesh,Back Office,2026-01-10,10,Saturday,09:58,19:55,0,5,9:57,9:57,P,10


## Execute Payroll Calculations

### Subtask:
Join attendance data with salary information and perform detailed payroll calculations including base salary, bonuses, and leverage adjustments.


**Reasoning**:
I will perform the payroll calculations by aggregating the attendance data, merging it with salary metadata, and applying the specified formulas for salary, bonuses, and deductions.



In [ ]:
import pandas as pd
import numpy as np

# 1. Aggregate normalized hours by emp_id and department
grouped_df = final_df.groupby(['emp_id', 'department'], as_index=False).agg(
    total_worked_hours=('normalized_work_hrs', 'sum')
)

# 2. Load salary metadata
salary_meta = pd.read_xlsx('/content/salary_bio.xlsx')

# 3. Merge attendance data with salary metadata
payroll_df = grouped_df.merge(salary_meta, on='emp_id', how='left')

# 4. Set constants (based on previous cell inputs or standard month)
HOURS_PER_DAY = 10
TOTAL_DAYS = 31 # Total days in Jan 2026
WEEKOFF_AND_PAID_LEAVE = 5 # Standard monthly leave/week-off

# 5. Calculate total expected hours
payroll_df['total_expected_hours'] = (TOTAL_DAYS - WEEKOFF_AND_PAID_LEAVE) * HOURS_PER_DAY
payroll_df.loc[payroll_df['total_expected_hours'] < 0, 'total_expected_hours'] = 0

# 6. Compute salary per hour
payroll_df['salary_per_hour'] = payroll_df['Salary'] / (TOTAL_DAYS * HOURS_PER_DAY)

# 7. Calculate base salary and fixed bonus
payroll_df['base_salary'] = payroll_df['total_worked_hours'] * payroll_df['salary_per_hour']
payroll_df['fixed_bonus'] = WEEKOFF_AND_PAID_LEAVE * (payroll_df['salary_per_hour'] * HOURS_PER_DAY)

# 8. Calculate leverage amount (capped between 0 and 3 hours)
leverage_diff = (payroll_df['total_expected_hours'] - payroll_df['total_worked_hours']).clip(lower=0, upper=3)
payroll_df['leverage_amount'] = leverage_diff * payroll_df['salary_per_hour']

# 9. Compute net salary
payroll_df['net_salary'] = payroll_df['base_salary'] + payroll_df['fixed_bonus'] + payroll_df['leverage_amount']

# 10. Apply deduction logic
conditions = [
    (payroll_df['Salary'] <= 21000),
    (payroll_df['Salary'] >= 21001) & (payroll_df['Salary'] <= 24999),
    (payroll_df['Salary'] >= 25000)
]
choices = [
    payroll_df['Salary'] * 0.0075, # ESI 0.75%
    0,                             # No deduction
    200                            # Fixed PD
]
payroll_df['deduction'] = np.select(conditions, choices, default=0)

# Final Payable
payroll_df['final_payable_salary'] = payroll_df['net_salary'] - payroll_df['deduction']

# Display the final report
report_cols = ['emp_id', 'emp_name', 'department', 'Salary', 'total_worked_hours', 'total_expected_hours', 'base_salary', 'fixed_bonus', 'leverage_amount', 'net_salary', 'deduction', 'final_payable_salary']
final_report = payroll_df[[c for c in report_cols if c in payroll_df.columns]]
print('Payroll calculations completed.')
display(final_report.head(10))

FileNotFoundError: [Errno 2] No such file or directory: '/content/salary_bio.csv'

# Task
Fix and execute the payroll calculations by modifying cell `87d5f3ec` to correctly load the salary metadata using `pd.read_excel('/content/salary_bio.xlsx')`. Ensure all calculation steps—including total expected hours, salary per hour, base salary, fixed bonus, leverage adjustments, and bracket-based deductions—are performed correctly. Finally, display the `final_report` to verify the employee IDs, salaries, and net payable amounts.

## Fix and Execute Payroll Calculations

### Subtask:
Correct the file loading logic in cell 87d5f3ec and execute the complete payroll calculation pipeline.


**Reasoning**:
I will correct the file path and format to read the salary metadata from the available Excel file, then execute the full payroll calculation logic as specified in the instructions.



In [ ]:
import pandas as pd
import numpy as np

# 1. Aggregate normalized hours by emp_id and department from final_df
grouped_df = final_df.groupby(['emp_id', 'department'], as_index=False).agg(
    total_worked_hours=('normalized_work_hrs', 'sum')
)

# 2. Load salary metadata from the correct Excel file path
salary_meta = pd.read_excel('/content/salary_bio.xlsx')

# 3. Merge attendance data with salary metadata
payroll_df = grouped_df.merge(salary_meta, on='emp_id', how='left')

# 4. Set constants for January 2026
HOURS_PER_DAY = 10
TOTAL_DAYS = 31
WEEKOFF_AND_PAID_LEAVE = 5

# 5. Calculate total expected hours
payroll_df['total_expected_hours'] = (TOTAL_DAYS - WEEKOFF_AND_PAID_LEAVE) * HOURS_PER_DAY
payroll_df.loc[payroll_df['total_expected_hours'] < 0, 'total_expected_hours'] = 0

# 6. Compute salary per hour
payroll_df['salary_per_hour'] = payroll_df['Salary'] / (TOTAL_DAYS * HOURS_PER_DAY)

# 7. Calculate base salary and fixed bonus
payroll_df['base_salary'] = payroll_df['total_worked_hours'] * payroll_df['salary_per_hour']
payroll_df['fixed_bonus'] = WEEKOFF_AND_PAID_LEAVE * (payroll_df['salary_per_hour'] * HOURS_PER_DAY)

# 8. Calculate leverage amount (capped between 0 and 3 hours)
leverage_diff = (payroll_df['total_expected_hours'] - payroll_df['total_worked_hours']).clip(lower=0, upper=3)
payroll_df['leverage_amount'] = leverage_diff * payroll_df['salary_per_hour']

# 9. Compute net salary
payroll_df['net_salary'] = payroll_df['base_salary'] + payroll_df['fixed_bonus'] + payroll_df['leverage_amount']

# 10. Apply deduction logic using np.select
conditions = [
    (payroll_df['Salary'] <= 21000),
    (payroll_df['Salary'] >= 21001) & (payroll_df['Salary'] <= 24999),
    (payroll_df['Salary'] >= 25000)
]
choices = [
    (payroll_df['Salary'] * 0.0075), # ESI 0.75%
    0,                              # No deduction
    200                             # Fixed PD
]
payroll_df['deduction'] = np.select(conditions, choices, default=0)

# 11. Final Payable Salary
payroll_df['final_payable_salary'] = payroll_df['net_salary'] - payroll_df['deduction']

# Display the final report
report_cols = ['emp_id', 'emp_name', 'department', 'Salary', 'total_worked_hours', 'total_expected_hours', 'base_salary', 'fixed_bonus', 'leverage_amount', 'net_salary', 'deduction', 'final_payable_salary']
final_report = payroll_df[[c for c in report_cols if c in payroll_df.columns]]
print('Payroll calculations completed successfully.')
final_report.head(10)

ValueError: You are trying to merge on object and int64 columns for key 'emp_id'. If you wish to proceed you should use pd.concat

**Reasoning**:
The merge failed because 'emp_id' in `grouped_df` is a string (object) while 'emp_id' in `salary_meta` is an integer. I will cast both columns to strings before merging to ensure compatibility.



In [ ]:
import pandas as pd
import numpy as np

# 1. Aggregate normalized hours by emp_id and department from final_df
grouped_df = final_df.groupby(['emp_id', 'department'], as_index=False).agg(
    total_worked_hours=('normalized_work_hrs', 'sum')
)

# 2. Load salary metadata from the correct Excel file path
salary_meta = pd.read_excel('/content/salary_bio.xlsx')

# Fix: Ensure emp_id columns are the same type (string) for merging
grouped_df['emp_id'] = grouped_df['emp_id'].astype(str)
salary_meta['emp_id'] = salary_meta['emp_id'].astype(str)

# 3. Merge attendance data with salary metadata
payroll_df = grouped_df.merge(salary_meta, on='emp_id', how='left')

# 4. Set constants for January 2026
HOURS_PER_DAY = 10
TOTAL_DAYS = 31
WEEKOFF_AND_PAID_LEAVE = 5

# 5. Calculate total expected hours
payroll_df['total_expected_hours'] = (TOTAL_DAYS - WEEKOFF_AND_PAID_LEAVE) * HOURS_PER_DAY
payroll_df.loc[payroll_df['total_expected_hours'] < 0, 'total_expected_hours'] = 0

# 6. Compute salary per hour
payroll_df['salary_per_hour'] = payroll_df['Salary'] / (TOTAL_DAYS * HOURS_PER_DAY)

# 7. Calculate base salary and fixed bonus
payroll_df['base_salary'] = payroll_df['total_worked_hours'] * payroll_df['salary_per_hour']
payroll_df['fixed_bonus'] = WEEKOFF_AND_PAID_LEAVE * (payroll_df['salary_per_hour'] * HOURS_PER_DAY)

# 8. Calculate leverage amount (capped between 0 and 3 hours)
leverage_diff = (payroll_df['total_expected_hours'] - payroll_df['total_worked_hours']).clip(lower=0, upper=3)
payroll_df['leverage_amount'] = leverage_diff * payroll_df['salary_per_hour']

# 9. Compute net salary
payroll_df['net_salary'] = payroll_df['base_salary'] + payroll_df['fixed_bonus'] + payroll_df['leverage_amount']

# 10. Apply deduction logic using np.select
conditions = [
    (payroll_df['Salary'] <= 21000),
    (payroll_df['Salary'] >= 21001) & (payroll_df['Salary'] <= 24999),
    (payroll_df['Salary'] >= 25000)
]
choices = [
    (payroll_df['Salary'] * 0.0075), # ESI 0.75%
    0,                              # No deduction
    200                             # Fixed PD
]
payroll_df['deduction'] = np.select(conditions, choices, default=0)

# 11. Final Payable Salary
payroll_df['final_payable_salary'] = payroll_df['net_salary'] - payroll_df['deduction']

# Display the final report
report_cols = ['emp_id', 'department', 'Salary', 'total_worked_hours', 'total_expected_hours', 'base_salary', 'fixed_bonus', 'leverage_amount', 'net_salary', 'deduction', 'final_payable_salary']
final_report = payroll_df[[c for c in report_cols if c in payroll_df.columns]]
print('Payroll calculations completed successfully.')
display(final_report.head(10))

Payroll calculations completed successfully.


,emp_id,department,Salary,total_worked_hours,total_expected_hours,base_salary,fixed_bonus,leverage_amount,net_salary,deduction,final_payable_salary
0,513,Back Office,20000,255,260,16451.612903,3225.806452,193.548387,19870.967742,150.0000,19720.967742
1,515,"HR, IT, Admin, Operations",18000,79,260,4587.096774,2903.225806,174.193548,7664.516129,135.0000,7529.516129
2,516,Sales,15999,274,260,14141.051613,2580.483871,0.000000,16721.535484,119.9925,16601.542984
3,527,Default Dept.,5000,0,260,0.000000,806.451613,48.387097,854.838710,37.5000,817.338710
4,538,Default Dept.,5000,152,260,2451.612903,806.451613,48.387097,3306.451613,37.5000,3268.951613
5,549,"HR, IT, Admin, Operations",5000,77,260,1241.935484,806.451613,48.387097,2096.774194,37.5000,2059.274194
6,550,Default Dept.,15000,263,260,12725.806452,2419.354839,0.000000,15145.161290,112.5000,15032.661290
7,567,Default Dept.,50030,70,260,11297.096774,8069.354839,484.161290,19850.612903,200.0000,19650.612903
8,577,Default Dept.,53000,146,260,24961.290323,8548.387097,512.903226,34022.580645,200.0000,33822.580645


## Verify Final Report

### Subtask:
Display and analyze the results in the `final_report` DataFrame to ensure all payroll components are accurately calculated and formatted.


## Summary:

### Q&A
**How were the payroll calculations fixed and executed?**
The process involved correcting the file path to `/content/salary_bio.xlsx`, ensuring data type consistency by casting `emp_id` to strings, and implementing a calculation pipeline that accounts for actual worked hours, fixed bonuses for paid leave (50 hours), and capped leverage adjustments (up to 3 hours).

**What deduction logic was applied to the final salaries?**
A bracket-based system was used:
*   Salaries $\le$ 21,000: 0.75% ESI deduction.
*   Salaries between 21,001 and 24,999: No deduction.
*   Salaries $\ge$ 25,000: Flat \$200 Professional Tax (PD) deduction.

### Data Analysis Key Findings
*   **Expected Hours**: Based on 31 total days in January 2026 and 5 days of week-offs/paid leave, the total expected working hours were set at 260 (10 hours/day).
*   **Base Salary Components**: The calculation correctly split compensation into `base_salary` (actual hours worked), `fixed_bonus` (equivalent to 50 hours of pay for leave), and a `leverage_amount` capped at 3 hours.
*   **Merge Success**: By normalizing `emp_id` to strings, the attendance data was successfully joined with the salary metadata, preventing previous merge errors.
*   **Financial Impact**: For an employee with a base salary of 20,000 (e.g., ID 513), the ESI deduction was calculated at \$150.00, resulting in a final payable amount of approximately \$19,720.97.

### Insights or Next Steps
*   **Verification**: Review employees with high `leverage_amount` values to ensure the 3-hour cap is sufficient for covering minor clock-in discrepancies.
*   **Automation**: The logic can now be wrapped into a function to process monthly payroll by simply updating the `TOTAL_DAYS` and `WEEKOFF_AND_PAID_LEAVE` constants.


In [ ]:
Common (Attendanace_Cleaning.ipynb, TPay.ipynb)
- total_expected_hours = (TOTAL_DAYS * HOURS_PER_DAY) - (WEEKOFF_AND_PAID_LEAVE * HOURS_PER_DAY)
- salary_per_hour = Salary / (TOTAL_DAYS * HOURS_PER_DAY)
- salary = total_worked_hours * salary_per_hour
-sundayandleave= WEEKOFF_AND_PAID_LEAVE * (salary_per_hour * HOURS_PER_DAY)
- leverage_diff = clamp(total_expected_hours - total_worked_hours, 0, 3)
- leverage_amount = leverage_diff * salary_per_hour
- net_salary = base_salary + fixed_bonus + leverage_amount
- deduction =
  - if Salary <= 21000: Salary * 0.75
  - if 21001 <= Salary <= 24999: 0
  - if Salary >= 25000: 200
- final_payable_salary = net_salary - deduction

SyntaxError: cannot assign to expression here. Maybe you meant '==' instead of '='? (1612649135.py, line 2)

In [ ]:
10 hrs 9 hrs 7.5 hrs
